# De la retina al tensor — preparar imágenes para entrenar una CNN

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stivenson/stivenson.github.io/blob/main/notebooks/glaucoma_preprocesado.ipynb)

Este notebook es el que acompaña al artículo [**De la retina al tensor**](https://stivenson.github.io/#/articles/imagen-a-tensor-cnn-glaucoma).
Contiene el mismo código, ejecutable de arriba a abajo: `Entorno de ejecución → Ejecutar todas`.

Vamos a convertir fotografías de fondo de ojo en el tensor que necesita una red
convolucional, y a medir por el camino **qué información sobrevive a cada paso** —
incluidas las pistas que permiten acertar el diagnóstico sin mirar la anatomía.

Los datos son de **ACRIMA** (Diaz-Pinto et al., 2019), 705 fondos de ojo etiquetados
como glaucomatosos o normales, publicados bajo CC BY 4.0.


## 0. Preparación: traer los datos

Igual que con un CSV, las imágenes se descargan desde una URL pública en vez de
subirlas a mano: lo que subes a Colab se borra al cerrar la sesión.

Se usan dos conjuntos. El **mini** (60 imágenes) para recorrer el pipeline, y
**ACRIMA completa** (705) para la sección de atajos, donde hacen falta todas.


In [ ]:
# Celda con la importación de librerías

import os                       # Rutas y tamaño de los archivos en disco
import io                       # Leer el zip descargado sin escribirlo antes
import zipfile                  # Descomprimir los conjuntos
import urllib.request           # Descarga por HTTP
from pathlib import Path        # Recorrer carpetas

import numpy as np              # Librería para trabajar con datos matriciales
from PIL import Image           # Decodificación de imágenes
import matplotlib.pyplot as plt # Librería para gráficas y visualización

# Establecer la semilla
np.random.seed(42)

print('Librerías listas')


In [ ]:
def descargar(url, destino):
    """
    Descarga un zip y lo descomprime, si no está ya en el disco.

    Parámetros de entrada:
    url     = dirección del archivo .zip
    destino = carpeta donde dejar las imágenes

    Parámetros de salida:
    destino = la misma carpeta, ya poblada
    """
    if os.path.isdir(destino) and os.listdir(destino):
        print(f'{destino}: ya estaba descargado')
        return destino

    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
        z.extractall('.')

    print(f'{destino}: {len(os.listdir(destino))} archivos')
    return destino


# El subconjunto de 60 imágenes que acompaña al artículo.
descargar('https://stivenson.github.io/datasets/acrima_mini.zip', 'acrima_mini')


In [ ]:
# ACRIMA completa (705 imágenes, ~24 MB) desde figshare, con DOI
# 10.6084/m9.figshare.7613135. Hace falta para la sección de atajos.
# Si figshare no responde, el notebook sigue con el subconjunto.

ACRIMA = 'acrima_mini'
try:
    with urllib.request.urlopen('https://ndownloader.figshare.com/files/14137700', timeout=180) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
        z.extractall('acrima_full')
    ACRIMA = 'acrima_full/Database/Images'
    print(f'ACRIMA completa: {len(os.listdir(ACRIMA))} imágenes')
except Exception as e:
    print(f'No se pudo bajar ACRIMA completa ({type(e).__name__}).')
    print('Se seguirá con el subconjunto de 60; las cifras cambiarán un poco.')

print(f'\nCarpeta para la sección de atajos: {ACRIMA}')


## 1. El archivo no es la imagen

Un `.jpg` no contiene píxeles: contiene instrucciones para reconstruirlos.
Es un formato de **compresión**, no de datos. Por eso `Image.open()` es
instantáneo — solo lee la cabecera. Los números aparecen al pedir el array.


In [ ]:
ruta = 'acrima_mini/Im318_g_ACRIMA.jpg'

img = Image.open(ruta)          # Todavía NO hay píxeles: solo la cabecera
print(f'Objeto perezoso  : {img}')
print(f'Bytes en disco   : {os.path.getsize(ruta):,}')

x = np.asarray(img)             # Aquí sí: se decodifica a una matriz de enteros
print(f'Bytes en memoria : {x.nbytes:,}')
print(f'Factor           : {x.nbytes / os.path.getsize(ruta):.1f}x')


Unas veintidós veces más grande al descomprimirse. Por eso no puedes cargar el
conjunto entero de golpe: las 705 imágenes ocupan 24 MB en disco, 684 MB como
enteros y 2,7 GB como `float32`.

> 💡 **Lección clave:** el peso del archivo no te dice cuánta memoria necesitas.
> Lo que importa es `alto × ancho × canales × bytes_por_valor`.


## 2. La imagen es un tensor

Lo que devuelve `np.asarray` es un tensor de tres dimensiones: alto, ancho y
canal. No hay colores ni bordes ni disco óptico: hay enteros de 0 a 255.


In [ ]:
print(f'shape : {x.shape}')
print(f'dtype : {x.dtype}')
print(f'rango : {x.min()} … {x.max()}')

# El píxel del centro, en las tres representaciones que verá el modelo.
cy, cx = x.shape[0] // 2, x.shape[1] // 2
MEAN = np.array([0.485, 0.456, 0.406], dtype='float32')
STD  = np.array([0.229, 0.224, 0.225], dtype='float32')

p = x[cy, cx].astype('float32')
print(f'\nuint8       : {x[cy, cx]}')
print(f'float [0,1] : {np.round(p / 255, 3)}')
print(f'normalizado : {np.round((p / 255 - MEAN) / STD, 3)}')


In [ ]:
def ver_pixeles(x, lado=12):
    """
    Amplía el centro de la imagen hasta que se ven los píxeles sueltos,
    con su valor escrito encima.

    Parámetros de entrada:
    x    = imagen como array (alto, ancho, 3)
    lado = número de píxeles de lado de la región ampliada

    Parámetros de salida:
    Gráfica con la imagen completa y la región ampliada.
    """
    cy, cx = x.shape[0] // 2, x.shape[1] // 2
    r = x[cy - lado // 2: cy + lado // 2, cx - lado // 2: cx + lado // 2]

    fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
    ax[0].imshow(x); ax[0].set_title(f'La imagen: {x.shape}'); ax[0].axis('off')
    ax[1].imshow(r, interpolation='nearest')
    ax[1].set_title(f'{lado}x{lado} píxeles del centro')
    ax[1].set_xticks([]); ax[1].set_yticks([])

    # El canal verde es el que mejor contrasta la estructura vascular.
    for i in range(r.shape[0]):
        for j in range(r.shape[1]):
            v = r[i, j]
            lum = 0.2126 * v[0] + 0.7152 * v[1] + 0.0722 * v[2]
            ax[1].text(j, i, f'{v[0]}\n{v[1]}\n{v[2]}', ha='center', va='center',
                       fontsize=6, color='black' if lum > 130 else 'white')
    plt.tight_layout()
    return plt.show()


ver_pixeles(x)


## 3. El conjunto real: dónde está la etiqueta

En MNIST la etiqueta viene servida. En ACRIMA está **en el nombre del archivo**:
lleva `_g_` si la imagen es patológica y solo `_` si es normal.

O sea que **la clase sana se marca por ausencia**. Esa es la primera trampa.


In [ ]:
import glob

# --- Lo que casi todo el mundo escribe la primera vez ---------------
rutas_mal = glob.glob('acrima_mini/*.jpg')
normales_mal = [r for r in rutas_mal if '_n_' in r]   # buscando la marca de 'normal'

print(f'Imágenes encontradas : {len(rutas_mal)}')
print(f'Etiquetadas normales : {len(normales_mal)}')
print('\nPero en la carpeta hay:', len(os.listdir('acrima_mini')))


Dos fallos a la vez, y ninguno lanza una excepción.

**Faltan dos imágenes.** Dos archivos tienen la extensión en mayúsculas (`.JPG`),
y `glob('*.jpg')` distingue mayúsculas en Linux — el sistema donde corre Colab.

**No hay ni una imagen normal.** Como `_n_` no existe en ningún nombre, el filtro
devuelve la lista vacía y `y` acabaría siendo de una sola clase.


In [ ]:
def cargar_rutas(carpeta):
    """
    Recoge las rutas de las imágenes y deduce la etiqueta del nombre del archivo.

    Parámetros de entrada:
    carpeta = ruta a la carpeta con las imágenes de ACRIMA

    Parámetros de salida:
    rutas = lista de rutas a las imágenes
    y     = np.array de enteros, 1 = glaucoma, 0 = normal
    """
    # Se filtra por sufijo en minúsculas: así entran tanto .jpg como .JPG.
    rutas = sorted(p for p in Path(carpeta).iterdir()
                   if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

    # La etiqueta es la PRESENCIA de '_g_'; lo normal se marca por ausencia.
    y = np.array([1 if '_g_' in p.name else 0 for p in rutas], dtype='int32')

    print(f'Imágenes encontradas    : {len(rutas)}')
    print(f'Datos por cada etiqueta : {np.bincount(y)}')
    print(f'Proporción de glaucoma  : {y.mean():.1%}')
    return rutas, y


rutas, y = cargar_rutas('acrima_mini')


> 💡 **Lección clave:** cuando la etiqueta vive en el nombre del archivo, el
> cargador de datos es código crítico. Cuenta siempre las clases justo después de
> construir `y` y compáralo con la documentación del conjunto.

Sobre ACRIMA completa esa proporción es 396 contra 309: **56,2 %**. Guárdalo.


## 4. La forma fija: por qué hay que redimensionar

Las 705 imágenes vienen en **258 tamaños distintos**, de 178×178 a 1420×1420.


In [ ]:
# Estas son las de la carpeta cargada ahora mismo (el subconjunto).
anchos = np.array([Image.open(r).size[0] for r in rutas])
print(f'en este subconjunto  : {len(np.unique(anchos))} tamaños, de {anchos.min()} a {anchos.max()}')
print(f'en ACRIMA completa   : 258 tamaños, de 178 a 1420')


Conviene ser preciso, porque hay dos exigencias distintas y solo una es negociable.

Las **capas convolucionales no necesitan un tamaño fijo**: un filtro de 3×3 se
desliza igual sobre 200 píxeles que sobre 1400. Una red totalmente convolucional
—con `GlobalAveragePooling2D` en lugar de `Flatten`— admite tamaños variables.

Lo que **nunca** es negociable es el **lote**: apilar 32 imágenes en un array obliga
a que las 32 compartan forma. Y en esta arquitectura concreta, además, el `Flatten`
fija la entrada de la capa densa que viene detrás.


In [ ]:
import tensorflow as tf         # Importar TensorFlow
from tensorflow import keras    # Importar la API Keras de TensorFlow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.layers import GlobalAveragePooling2D

# Establecer la semilla para el generador de TensorFlow
tf.random.set_seed(42)

print('TensorFlow', tf.__version__)


In [ ]:
def create_model(name, entrada, n_clases, cabeza='flatten'):
    """
    Construye la misma arquitectura convolucional para tamaños de entrada
    distintos, para poder comparar el número de parámetros.

    Parámetros:
    name     = nombre del modelo
    entrada  = tupla (alto, ancho, canales)
    n_clases = número de neuronas de salida
    cabeza   = 'flatten' o 'gap'

    Salida:
    model = modelo neuronal convolucional en tensorflow.keras
    """
    model = Sequential(name=name)
    model.add(Input(shape=entrada))

    # Primera capa convolucional: 16 filtros con kernel de 3x3.
    # Parámetros = 3*3*canales_entrada*16 + 16 (un sesgo por filtro).
    model.add(Conv2D(16, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Segunda capa: 32 filtros de 3x3 sobre los 16 canales anteriores.
    # Parámetros = 3*3*16*32 + 32 = 4640, sea cual sea el tamaño de entrada.
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D(pool_size=(4, 4)))

    # Aquí se decide si el modelo depende del tamaño de entrada o no.
    if cabeza == 'flatten':
        model.add(Flatten())
    else:
        model.add(GlobalAveragePooling2D())

    model.add(Dense(100, activation='relu'))
    model.add(Dense(n_clases, activation='softmax'))
    return model


for etiqueta, entrada, n in [('MNIST ', (28, 28, 1), 10),
                             ('Retina', (224, 224, 3), 2)]:
    m = create_model(etiqueta.strip(), entrada, n)
    conv = sum(c.count_params() for c in m.layers if isinstance(c, Conv2D))
    print(f'{etiqueta} {str(entrada):16s} conv={conv:>6,}  total={m.count_params():>10,}')


**La misma arquitectura, letra por letra.** Solo cambia el tamaño de entrada, y
los parámetros pasan de 34.710 a 2.514.190: **72 veces más**. Las convolucionales
apenas notan el cambio (4.800 y 5.088); los 2,5 millones están todos en la capa
densa que alimenta el `Flatten`.

Y mientras MNIST trae 60.000 imágenes de entrenamiento, ACRIMA trae 705. Eso no
demuestra que el modelo vaya a fallar, pero sí que tiene margen de sobra para
memorizar en lugar de aprender. **Riesgo altísimo de sobreajuste**, que se confirma
mirando las curvas y evaluando en una partición independiente.

La salida no es entrenar más rato:


In [ ]:
# Cambiar Flatten por GlobalAveragePooling2D promedia cada mapa de activación y
# devuelve un vector de longitud igual al número de filtros (32), sin depender
# del tamaño de entrada.
gap = create_model('Retina_GAP', (224, 224, 3), 2, cabeza='gap')
flat = create_model('Retina_Flatten', (224, 224, 3), 2, cabeza='flatten')

print(f'con Flatten                : {flat.count_params():>10,}')
print(f'con GlobalAveragePooling2D : {gap.count_params():>10,}')
print(f'reducción                  : {flat.count_params() / gap.count_params():.0f}x')


### Cómo se reduce importa

Bajar de 1420×1420 a 224×224 tira el **97,5 %** de los píxeles. La forma de tirarlos
no da igual — y aquí hay una trampa de nombres que conviene no repetir.

`tf.image.resize` usa **`bilinear` por defecto**, y `antialias=True` **no lo cambia**
**a `area`**: ensancha el filtro de muestreo del método que hayas elegido. La
documentación dice además que con `area` ese argumento *no tiene ningún efecto*.


In [ ]:
grande = np.asarray(Image.open(rutas[int(np.argmax(anchos))]).convert('RGB'))
print(f'Imagen de partida: {grande.shape}')

variantes = {
    'nearest (sin promediar)': tf.image.resize(grande, [224, 224], method='nearest'),
    'bilinear (por defecto)':  tf.image.resize(grande, [224, 224]),
    'bilinear + antialias':    tf.image.resize(grande, [224, 224], antialias=True),
    'area (hay que pedirlo)':  tf.image.resize(grande, [224, 224], method='area'),
}

fig, ax = plt.subplots(1, 4, figsize=(16, 4.4))
for a, (nombre, v) in zip(ax, variantes.items()):
    a.imshow(np.asarray(v).astype('uint8'))
    a.set_title(nombre, fontsize=10)
    a.axis('off')
plt.tight_layout(); plt.show()


## 5. El rango: normalizar, y dónde se calcula

Son dos operaciones distintas y conviene no confundirlas.

**Dividir entre 255 es seguro**: 255 es el techo del tipo `uint8`, no una
estadística de tus datos. **Calcular media y desviación sobre el conjunto entero,
en cambio, es medir tus datos** — y si lo haces antes de partir, las estadísticas
de tus imágenes de prueba entran en el preprocesado del entrenamiento. Eso es fuga.


In [ ]:
# Paso 1 — escalar a [0, 1].
z = np.asarray(Image.open(ruta).convert('RGB').resize((224, 224)), dtype='float32')
z = z / 255.0

# Paso 2 — centrar y tipificar. OJO: estas constantes son las de torchvision,
# la convención de PyTorch. NO son universales (ver la celda siguiente).
z = (z - MEAN) / STD

print(f'rango: {z.min():.2f} … {z.max():.2f}   media: {z.mean():.3f}')


**No hay un preprocesado estándar.** Keras no usa esas constantes por defecto.
Cada familia de modelos trae su propia `preprocess_input`, y conviven tres modos:

| Modo | Qué hace | Rango | Ejemplo |
|---|---|---|---|
| `caffe` | RGB→BGR y resta `[103.939, 116.779, 123.68]`. **No divide entre 255** | ≈ −124 … 151 | ResNet50, VGG16 |
| `tf` | Escala a `[-1, 1]` | −1 … 1 | MobileNet, EfficientNet |
| `torch` | Escala a `[0,1]` y tipifica con la media y desviación de arriba | ≈ −2,1 … 2,6 | DenseNet |

`ResNet50` usa **`caffe`**, que es el modo por defecto:


In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

# preprocess_input de ResNet50 espera la imagen CRUDA en [0, 255]:
# ya se encarga él de pasar a BGR y restar la media. No hay que dividir antes.
cruda = np.asarray(Image.open(ruta).convert('RGB').resize((224, 224)), dtype='float32')

x_bien = preprocess_input(cruda.copy())           # como se espera
x_mal  = preprocess_input(cruda.copy() / 255.0)   # ya dividida: preprocesada dos veces

for nombre, v in [('bien', x_bien), ('mal ', x_mal)]:
    print(f'{nombre} : {v.min():8.2f} … {v.max():8.2f}   amplitud {v.max() - v.min():7.2f}')


Fíjate en la amplitud: pasa de **255 a 20**. Al haber dividido antes, toda la imagen
entra en `[0, 1]` y lo único que hace `preprocess_input` es restarle la media, así
que las tres bandas se apilan en una franja estrecha y negativa.

Keras no protesta. El entrenamiento arranca, la pérdida baja un poco y se estanca.

> 💡 **Lección clave:** los errores de rango no lanzan excepciones. Imprime `min`,
> `max` y `media` de un lote justo antes de `fit()`.


## 6. El eje de canal, y la cuarta dimensión

Keras trabaja en **NHWC** (`channels_last`): lote, alto, ancho, canal. Este es el
motivo de esa línea que aparece en todos los laboratorios de MNIST:


In [ ]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()
print(f'MNIST tal como llega : {X_train.shape}   <- sin eje de canal')

# El resultado hay que guardarlo: reshape NO modifica el array original.
X_train = X_train.reshape((60000, 28, 28, 1))
print(f'Con el eje de canal  : {X_train.shape}')

# Añadir un eje de longitud 1 no cambia ni un valor. Sobre un array contiguo
# NumPy devuelve una vista; en general reshape copia si no puede describir la
# nueva forma sobre la memoria existente.
print(f'¿Comparte memoria?   : {np.shares_memory(X_train, X_train.reshape(60000, 784))}')


In [ ]:
def preparar(ruta, lado=224):
    """
    Aplica el pipeline completo a una imagen y devuelve su tensor.

    Parámetros de entrada:
    ruta = ruta a la imagen
    lado = tamaño de salida, en píxeles

    Parámetros de salida:
    x = np.ndarray (lado, lado, 3) float32, escalado y tipificado
    """
    img = Image.open(ruta).convert('RGB').resize((lado, lado))
    x = np.asarray(img, dtype='float32') / 255.0
    return (x - MEAN) / STD


lote = np.stack([preparar(r) for r in rutas[:32]])   # 32 imágenes ya preparadas

print(f'Tamaño del lote : {lote.shape}')
print(f'Tipo            : {lote.dtype}')
print(f'Memoria         : {lote.nbytes / 1024**2:.1f} MiB')


18,4 MiB para 32 imágenes —19,3 MB si cuentas en potencias de diez— y eso es solo
**la entrada**. Cada capa guarda su mapa de activaciones para la retropropagación,
así que la memoria real durante el entrenamiento es varias veces esa cifra.

Por eso el tamaño de lote es lo primero que se baja cuando la GPU se queda sin
memoria — y acaba decidiéndolo el tamaño al que redimensionaste, tres pasos antes.


---

## Hallazgo 1 — Por qué un accuracy alto no basta

Antes de celebrar cualquier cifra, la comparación obligatoria: el **clasificador**
**tonto**, el que ignora la imagen y responde siempre la clase mayoritaria.

Pero el suelo verdadero está mucho más arriba.


In [ ]:
rutas_all, y_all = cargar_rutas(ACRIMA)

anchos_all = np.array([Image.open(r).size[0] for r in rutas_all])
pesos_all  = np.array([os.path.getsize(r) for r in rutas_all])

print(f'\nancho medio, glaucoma : {anchos_all[y_all == 1].mean():.1f} px')
print(f'ancho medio, normal   : {anchos_all[y_all == 0].mean():.1f} px')


In [ ]:
def color_medio(rutas):
    """
    Diferencia media entre el canal rojo y el azul de cada imagen. Es un solo
    número por imagen: no mira ninguna estructura anatómica.
    """
    out = []
    for r in rutas:
        a = np.asarray(Image.open(r).convert('RGB').resize((64, 64)), dtype='float32')
        out.append(a[..., 0].mean() - a[..., 2].mean())
    return np.array(out)


def mejor_umbral(v, y):
    """Umbral que maximiza el acierto de la regla "v >= t", probando el signo."""
    mejor = (0.0, None, 1)
    for t in np.unique(v):
        for signo in (1, -1):
            acc = float((((v - t) * signo >= 0).astype('int32') == y).mean())
            if acc > mejor[0]:
                mejor = (acc, float(t), signo)
    return mejor


colores_all = color_medio(rutas_all)
tonto = np.bincount(y_all).max() / len(y_all)

print(f"{'atajo':<24}{'acierto':>10}")
print('-' * 34)
print(f"{'clasificador tonto':<24}{tonto:>10.1%}")
for nombre, v in [('ancho de la imagen', anchos_all),
                  ('peso del archivo', pesos_all),
                  ('color medio (R - B)', colores_all)]:
    acc, t, signo = mejor_umbral(v, y_all)
    print(f'{nombre:<24}{acc:>10.1%}   umbral {t:,.0f}')


**Sin abrir la imagen, sin ver un vaso, sin saber qué es un disco óptico.**

Las glaucomatosas de ACRIMA miden 668 px de lado de media; las normales, 353. Las
dos clases se capturaron o recortaron de formas sistemáticamente distintas, y eso
quedó grabado en las dimensiones del archivo.

Ese umbral se elige mirando las mismas imágenes que luego se puntúan, así que
sobreestima. La objeción es justa y se responde midiendo:


In [ ]:
def held_out(v, y, repeticiones=200, frac=0.70, semilla=42):
    """
    Ajusta el umbral en una parte y lo evalúa en la otra, estratificado.
    Devuelve la media y la desviación del acierto en el conjunto de prueba.
    """
    rng = np.random.default_rng(semilla)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    aciertos = []
    for _ in range(repeticiones):
        tr = np.concatenate([rng.permutation(idx)[:int(round(frac * len(idx)))]
                             for idx in (pos, neg)])
        m = np.zeros(len(y), dtype=bool); m[tr] = True
        _, t, signo = mejor_umbral(v[m], y[m])
        pred = ((v[~m] - t) * signo >= 0).astype('int32')
        aciertos.append(float((pred == y[~m]).mean()))
    return float(np.mean(aciertos)), float(np.std(aciertos))


print(f"{'atajo':<24}{'in-sample':>11}{'held-out':>16}")
print('-' * 51)
for nombre, v in [('ancho de la imagen', anchos_all),
                  ('peso del archivo', pesos_all),
                  ('color medio (R - B)', colores_all)]:
    acc_in, _, _ = mejor_umbral(v, y_all)
    mu, sd = held_out(v, y_all)
    print(f'{nombre:<24}{acc_in:>10.1%}{mu:>12.1%} ± {sd:.1%}')


Los atajos **sobreviven** a la partición independiente: no eran un artefacto de
buscar el umbral sobre los mismos datos.

Y aquí es donde el preprocesado deja de ser fontanería. **Redimensionar le quita a**
**la red el acceso** al ancho y al peso: cuando todo llega como tensor de 224×224,
esa información ya no está en lo que el modelo ve. El paso que parecía trámite es
la defensa principal contra el atajo más fuerte del conjunto.

Conviene no exagerarlo: borra el acceso directo, no toda la huella. Una imagen que
venía de 1420 px y otra de 300 llegan a 224 con nitidez y artefactos de compresión
distintos. Y **el color sobrevive sin más**.

> 💡 **Lección clave:** el preprocesado decide a qué atajos les quitas el acceso.
> Antes de creerte una métrica, mide qué acierta un modelo trivial que solo vea los
> metadatos. Si tu CNN no supera eso por un margen amplio, no has demostrado nada.


## Hallazgo 2 — La partición que infla la nota

Hay un segundo atajo, y no vive en las imágenes sino en cómo las repartes.

ACRIMA distribuye imágenes anónimas sin identificador de paciente, así que aquí no
se puede medir. Pero es el fallo más caro en imagen médica, y conjuntos como
**PAPILA** —los dos ojos de cada uno de sus 244 pacientes— existen para evitarlo.


In [ ]:
def simular_particion(n_pacientes=1000, p_train=0.8, semilla=42):
    """
    Reparte los dos ojos de cada paciente y cuenta cuántos quedan partidos.

    Un paciente queda partido cuando sus dos ojos caen a lados distintos, lo
    que ocurre con probabilidad 2p(1-p).
    """
    rng = np.random.default_rng(semilla)
    ojos = rng.random((n_pacientes, 2)) < p_train      # True = entrenamiento
    partidos = (ojos[:, 0] != ojos[:, 1]).mean()

    # Para una imagen de prueba, ¿su gemela está en entrenamiento?
    en_prueba = ~ojos
    gemela_en_train = (en_prueba & ojos[:, ::-1]).sum() / en_prueba.sum()
    return partidos, gemela_en_train


for p in (0.5, 0.8):
    partidos, gemela = simular_particion(p_train=p)
    print(f'reparto {round(p*100)}/{round((1-p)*100)}  ->  '
          f'2p(1-p) = {2*p*(1-p):.2f}   pacientes partidos: {partidos:.1%}   '
          f'imágenes de prueba con su gemela en entrenamiento: {gemela:.1%}')


Los dos ojos de una persona se fotografían el mismo día, con la misma cámara, y
comparten pigmentación, calibre de vasos y buena parte de la anatomía. Reconocer el
segundo después de haber visto el primero no es diagnosticar: es recordar.

Con un reparto 80/20, **el 80 % de tus imágenes de prueba tiene a su gemela dentro**
**del entrenamiento**. Al partir por paciente, cero.

La regla, sin excepciones: **agrupa por la unidad que quieres que el modelo**
**generalice**. Si vas a diagnosticar personas, parte por persona.


---

## Resumen

| Paso | Lo que parece | Lo que de verdad decide |
|---|---|---|
| Decodificar | Abrir el archivo | El factor ×22 de memoria que define tu tamaño de lote |
| Redimensionar | Ajustar el tamaño | La capacidad del modelo (72×) y **a qué atajos les quitas el acceso** |
| Interpolar | Un detalle | Si los vasos finos sobreviven o se convierten en ruido |
| Escalar `/255` | Un trámite | Nada. Es seguro: 255 es una constante del tipo |
| Normalizar | Un trámite | Una fuga, si calculas las estadísticas antes de partir |
| Eje de canal | Ceremonia de Keras | Lo único de esta lista que falla ruidosamente |
| Apilar el lote | Eficiencia | Tu límite de memoria en GPU |
| Partir | Lo de siempre | Si tu métrica significa algo |

**Una CNN no ve una imagen, ve un tensor.** Todo lo que hay entre el archivo y ese
tensor es donde eliges qué información llega al modelo, y qué caminos fáciles le
dejas para acertar sin aprender.

El artículo completo, con las visualizaciones interactivas: [**De la retina al tensor**](https://stivenson.github.io/#/articles/imagen-a-tensor-cnn-glaucoma).

---

*Datos: ACRIMA (Diaz-Pinto et al., 2019, [10.1186/s12938-019-0649-y](https://doi.org/10.1186/s12938-019-0649-y))*
*y HRF (Budai et al., 2013), ambos bajo CC BY 4.0.*
